# 1조 리서치 센터: 내 지시서로 리포트 한 번 끝까지 만들기

1. 아래 첫 코드 칸의 `MEMBER`(내 파트: industry, company, valuation, center), `STOCK_CODE`(종목코드), `YEAR`(사업보고서 연도)만 바꿔요.
2. 왼쪽 🔑 Secrets에 `DART_API_KEY`, `GEMINI_API_KEY`가 있고 노트북 액세스가 켜져 있어야 해요.
3. 런타임 → 모두 실행을 눌러요.

Colab은 GitHub에 올라가 있는 지시서를 읽어요. 내 PC에서 고친 지시서는 GitHub에 올린 뒤에야 쓰여요. 실행할 때마다 최신 내용을 받아 와요(`git pull`).

지시서 6개는 `members/내 파트/prompts/`에 있는 파일을 먼저 쓰고, 없는 파일은 공통 `prompts/`를 써요. 결과 맨 위 "지시서" 줄에서 각 지시서를 어느 파일에서 읽었는지 볼 수 있어요.

모두 실행할 때마다 새 결과 폴더(`members/내 파트/runs/종목코드_연도_월일_시분`)를 만들고, 같은 종목·연도의 직전 폴더에서 첫 바퀴 답을 가져와요. 그래서 바꾸지 않은 파트는 Gemini를 다시 부르지 않아요(♻). 중간에 멈추면 마지막 칸만 다시 실행하세요.

Colab의 결과 폴더는 런타임이 끝나면 사라져요. 모임에 가져갈 `report.md`, `report.html`, `verification.md`는 왼쪽 파일 창에서 내려받아 두세요.

In [ ]:
MEMBER = "industry"
STOCK_CODE = "004370"
YEAR = 2025

In [ ]:
!pip -q install google-genai pyyaml
import os, sys
if not os.path.exists("/content/kfa-1jo-research"):
    !git clone -q https://github.com/phm0305/kfa-1jo-research /content/kfa-1jo-research
os.chdir("/content/kfa-1jo-research")
!git pull -q
sys.path[:0] = ["engine", "research-center"]
from datetime import datetime
from collect import KST
from pipeline import new_run_dir
RUNS = f"members/{MEMBER}/runs"
RUN_DIR = new_run_dir(RUNS, STOCK_CODE, YEAR, datetime.now(KST))

In [ ]:
from IPython.display import Markdown, display
from collect import get_key
from pipeline import run

result = run(STOCK_CODE, YEAR, [f"members/{MEMBER}/prompts", "prompts"], RUN_DIR,
             get_key("GEMINI_API_KEY"), get_key("DART_API_KEY"), quota_dirs=[RUNS])
for step, mark, memo in result.status:
    print(mark, step, memo)
if result.verification:
    display(Markdown(result.verification.read_text(encoding="utf-8")))
if result.report:
    display(Markdown(result.report.read_text(encoding="utf-8")))